In [ ]:
# Import required modules
import sys
sys.path.append('..')

from prioritization import (
    load_plants,
    validate_plants,
    prioritize_plants,
    export_all_formats,
    print_report_summary,
)
from prioritization.models import PriorityLevel
from prioritization.prioritize import get_summary_statistics

import json
from pathlib import Path

## 1. Load Plant Data

Load plant data from the Firecrawl extraction output or other sources.

In [ ]:
# Load from Jupyter notebook output
plants = load_plants('../main.ipynb', source_type='notebook')

print(f"Loaded {len(plants)} plants")
print(f"\nFirst plant: {plants[0].scientific_name}")

## 2. Validate Plant Data

Filter plants based on minimum requirements for scoring.

In [ ]:
# Validate with minimum requirements
min_requirements = {
    'min_pathogens': 1,
    'min_mic_values': 1,
}

valid_plants = validate_plants(plants, min_requirements)

print(f"Valid plants: {len(valid_plants)}")
print(f"Excluded: {len(plants) - len(valid_plants)} plants")

## 3. Run Prioritization

Calculate prioritization scores for all valid plants.

In [ ]:
# Run prioritization
report = prioritize_plants(valid_plants)

print(f"\nPrioritization Complete!")
print(f"Total plants analyzed: {report.total_plants}")
print(f"\nPriority Distribution:")
print(f"  High:   {report.high_priority_count} plants")
print(f"  Medium: {report.medium_priority_count} plants")
print(f"  Low:    {report.low_priority_count} plants")
print(f"\nAverage Score: {report.average_score:.3f}")
print(f"Median Score:  {report.median_score:.3f}")

## 4. View Top Plants

Display the top-ranked plants with detailed scores.

In [ ]:
# Print top 10 plants
print("\n" + "="*80)
print("TOP 10 PLANTS")
print("="*80 + "\n")

for plant in report.get_top_n(10):
    priority_emoji = {
        PriorityLevel.HIGH: "🔴",
        PriorityLevel.MEDIUM: "🟡",
        PriorityLevel.LOW: "⚪",
    }
    
    print(f"{plant.rank}. {priority_emoji[plant.priority_level]} {plant.scientific_name}")
    if plant.local_name:
        print(f"   Local Name: {plant.local_name}")
    print(f"   Total Score: {plant.total_score:.3f} ({plant.priority_level.value.upper()})")
    print(f"   Percentile: {plant.percentile:.1f}%")
    print(f"   Category Scores:")
    print(f"     - Ethnobotanical: {plant.ethnobotanical_score.score:.3f}")
    print(f"     - Antimicrobial:  {plant.antimicrobial_score.score:.3f}")
    print(f"     - Safety:         {plant.safety_score.score:.3f}")
    print(f"     - Feasibility:    {plant.feasibility_score.score:.3f}")
    print(f"   Metrics: {plant.pathogen_count} pathogens, {plant.mic_count} MIC values")
    print()

## 5. Analyze High Priority Plants

Extract and analyze plants classified as high priority.

In [ ]:
# Get high priority plants
high_priority = report.get_high_priority_plants()

print(f"\nHIGH PRIORITY PLANTS ({len(high_priority)})")
print("="*80 + "\n")

for plant in high_priority:
    print(f"{plant.rank}. {plant.scientific_name}")
    print(f"   Score: {plant.total_score:.3f}")
    print(f"   Antimicrobial: {plant.antimicrobial_score.justification}")
    print(f"   WHO Priority Pathogens: {plant.who_priority_pathogens}")
    print(f"   AMR Strains: {plant.amr_strain_count}")
    print()

## 6. Filter by Criteria

Find plants meeting specific research criteria.

In [ ]:
# Find high priority plants with WHO priority pathogens
who_priority_plants = [
    p for p in report.plant_scores
    if p.priority_level == PriorityLevel.HIGH
    and p.who_priority_pathogens > 0
]

print(f"\nHigh Priority Plants with WHO Priority Pathogens: {len(who_priority_plants)}")
print("="*80 + "\n")

for plant in who_priority_plants[:5]:
    print(f"{plant.scientific_name}")
    print(f"  Score: {plant.total_score:.3f}")
    print(f"  WHO Priority Pathogens: {plant.who_priority_pathogens}")
    print(f"  Antimicrobial Details:")
    if 'average_mic' in plant.antimicrobial_score.details:
        print(f"    - Avg MIC: {plant.antimicrobial_score.details['average_mic']} μg/mL")
    if 'unique_pathogens' in plant.antimicrobial_score.details:
        print(f"    - Unique Pathogens: {plant.antimicrobial_score.details['unique_pathogens']}")
    print()

## 7. Compare Two Plants

Side-by-side comparison of scoring categories.

In [ ]:
from prioritization.prioritize import compare_plants

# Compare top 2 plants
if len(report.plant_scores) >= 2:
    plant1 = report.plant_scores[0]
    plant2 = report.plant_scores[1]
    
    comparison = compare_plants(plant1, plant2)
    
    print("\nPLANT COMPARISON")
    print("="*80)
    print(f"\nPlant 1: {comparison['plant1']['name']}")
    print(f"  Total Score: {comparison['plant1']['total_score']:.3f}")
    print(f"  Rank: {comparison['plant1']['rank']}")
    print(f"  Priority: {comparison['plant1']['priority'].upper()}")
    
    print(f"\nPlant 2: {comparison['plant2']['name']}")
    print(f"  Total Score: {comparison['plant2']['total_score']:.3f}")
    print(f"  Rank: {comparison['plant2']['rank']}")
    print(f"  Priority: {comparison['plant2']['priority'].upper()}")
    
    print("\nDifferences (Plant 1 - Plant 2):")
    for category, diff in comparison['differences'].items():
        print(f"  {category:20s}: {diff:+.3f}")
    
    print("\nCategory Winners:")
    for category, winner in comparison['winner'].items():
        print(f"  {category:20s}: {winner}")

## 8. Summary Statistics

Comprehensive statistical analysis of the prioritization results.

In [ ]:
# Get detailed statistics
stats = get_summary_statistics(report)

print("\nSUMMARY STATISTICS")
print("="*80)
print(json.dumps(stats, indent=2))

## 9. Export Results

Save prioritization results in multiple formats.

In [ ]:
# Create output directory
output_dir = Path('prioritization_results')
output_dir.mkdir(exist_ok=True)

# Export all formats
output_files = export_all_formats(report, output_dir)

print("\nEXPORTED FILES")
print("="*80)
for fmt, path in output_files.items():
    print(f"{fmt:15s}: {path}")

## 10. Detailed Plant Analysis

Deep dive into the scoring breakdown for the top plant.

In [ ]:
# Analyze top plant in detail
top_plant = report.plant_scores[0]

print(f"\nDETAILED ANALYSIS: {top_plant.scientific_name}")
print("="*80)

print(f"\nOverall:")
print(f"  Rank: #{top_plant.rank} out of {report.total_plants}")
print(f"  Total Score: {top_plant.total_score:.3f}")
print(f"  Percentile: Top {100 - top_plant.percentile:.1f}%")
print(f"  Priority: {top_plant.priority_level.value.upper()}")

print(f"\nEthnobotanical Score: {top_plant.ethnobotanical_score.score:.3f}")
print(f"  Weight: {top_plant.ethnobotanical_score.weight}")
print(f"  Weighted Score: {top_plant.ethnobotanical_score.weighted_score:.3f}")
print(f"  Justification: {top_plant.ethnobotanical_score.justification}")
print(f"  Details:")
if 'sub_scores' in top_plant.ethnobotanical_score.details:
    for key, value in top_plant.ethnobotanical_score.details['sub_scores'].items():
        print(f"    {key}: {value:.3f}")

print(f"\nAntimicrobial Score: {top_plant.antimicrobial_score.score:.3f}")
print(f"  Weight: {top_plant.antimicrobial_score.weight}")
print(f"  Weighted Score: {top_plant.antimicrobial_score.weighted_score:.3f}")
print(f"  Justification: {top_plant.antimicrobial_score.justification}")
print(f"  Details:")
print(f"    Pathogen Count: {top_plant.antimicrobial_score.details.get('pathogen_count', 0)}")
print(f"    MIC Count: {top_plant.antimicrobial_score.details.get('mic_count', 0)}")
print(f"    WHO Priority Pathogens: {top_plant.antimicrobial_score.details.get('who_priority_pathogens', 0)}")
print(f"    AMR Strains: {top_plant.antimicrobial_score.details.get('amr_strain_count', 0)}")
if 'average_mic' in top_plant.antimicrobial_score.details:
    print(f"    Average MIC: {top_plant.antimicrobial_score.details['average_mic']} μg/mL")

print(f"\nSafety Score: {top_plant.safety_score.score:.3f}")
print(f"  Justification: {top_plant.safety_score.justification}")

print(f"\nFeasibility Score: {top_plant.feasibility_score.score:.3f}")
print(f"  Justification: {top_plant.feasibility_score.justification}")

## Conclusion

This notebook demonstrated the complete workflow for prioritizing medicinal plants using the comprehensive scoring algorithm. The results can be used to:

1. Identify high-priority plants for antimicrobial research
2. Focus on plants with activity against WHO priority pathogens
3. Balance efficacy, safety, traditional knowledge, and feasibility
4. Generate reports for research planning and grant proposals

Next steps:
- Review high-priority plants for wet lab testing
- Plan collection expeditions in Nepal
- Collaborate with traditional healers for additional insights
- Conduct toxicity and safety assessments